
|dept| name |salary|
|----|------|----
| IT |Alice | 100  |
| IT | Bob  | 120  |
| IT |Carol | 120  |
| HR | Dan  |  90  |
| HR | Eve  | 110  |

In [25]:
import pandas as pd
import numpy as np
import pyspark.sql.functions as SF
from pyspark.sql import SparkSession, Window

In [13]:
test_data = [
    ("IT", "Alice", 100),
    ("IT", "Bob", 120),
    ("IT", "Carol", 120),
    ("HR", "Dan", 90),
    ("HR", "Eve", 110)
]

In [5]:
data = pd.DataFrame(test_data, columns=["dept", "name", "salary"])

data

,dept,name,salary
0,IT,Alice,100
1,IT,Bob,120
2,IT,Carol,120
3,HR,Dan,90
4,HR,Eve,110


In [6]:
spark = (
    SparkSession.builder
    .appName("test_data")
    .getOrCreate()
)
print(spark.sparkContext.uiWebUrl)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/02/09 14:41:10 WARN Utils: Your hostname, void, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/02/09 14:41:10 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/09 14:41:11 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


http://10.255.255.254:4040


In [14]:
df_spark = spark.createDataFrame(test_data,
                                 schema="dept: string, name: string, salary: int")

df_spark.show()

+----+-----+------+
|dept| name|salary|
+----+-----+------+
|  IT|Alice|   100|
|  IT|  Bob|   120|
|  IT|Carol|   120|
|  HR|  Dan|    90|
|  HR|  Eve|   110|
+----+-----+------+



In [15]:
df_spark.show()
data

+----+-----+------+
|dept| name|salary|
+----+-----+------+
|  IT|Alice|   100|
|  IT|  Bob|   120|
|  IT|Carol|   120|
|  HR|  Dan|    90|
|  HR|  Eve|   110|
+----+-----+------+



,dept,name,salary
0,IT,Alice,100
1,IT,Bob,120
2,IT,Carol,120
3,HR,Dan,90
4,HR,Eve,110


### Подзапрос

In [ ]:
"""
    SELECT e.dept, e.name, e.salary
    FROM employees e
    JOIN (
        SELECT dept, AVG(salary) AS avg_salary
        FROM employees
        GROUP BY dept
    ) a
    ON e.dept = a.dept
    WHERE e.salary > a.avg_salary;
"""


avg_salary = (

    df_spark
    .groupBy("dept")
    .agg(SF.mean("salary").alias("avg_salary"))
)

res = (
    df_spark
    .join(avg_salary, "dept")
    .where(SF.col("salary") > SF.col("avg_salary"))
)

res.show()

+----+-----+------+------------------+
|dept| name|salary|        avg_salary|
+----+-----+------+------------------+
|  IT|  Bob|   120|113.33333333333333|
|  IT|Carol|   120|113.33333333333333|
|  HR|  Eve|   110|             100.0|
+----+-----+------+------------------+



### Оконная функция

In [ ]:
"""
    SELECT
        dept,
        name,
        salary,
        AVG(salary) OVER (PARTITION BY dept) AS avg_salary
    FROM employees
    WHERE salary > AVG(salary) OVER (PARTITION BY dept);

или, потому что (B реальном SQL это часто оборачивают в подзапрос, т.к. WHERE не всегда принимает оконные функции)

    SELECT *
    FROM (
        SELECT
            dept,
            name,
            salary,
            AVG(salary) OVER (PARTITION BY dept) AS avg_salary
        FROM employees
    ) t
    WHERE salary > avg_salary;


"""

res_win = (
    df_spark
    .withColumn(
        "avg_salary", SF.mean("salary")
        .over(Window.partitionBy("dept"))
    )
    .where(SF.col("salary") > SF.col("avg_salary"))
)

res_win.show()

+----+-----+------+------------------+
|dept| name|salary|        avg_salary|
+----+-----+------+------------------+
|  HR|  Eve|   110|             100.0|
|  IT|  Bob|   120|113.33333333333333|
|  IT|Carol|   120|113.33333333333333|
+----+-----+------+------------------+



In [35]:
spark.stop()